# Estimations — country fixed effects, year fixed effects and Common Correlated Effects (CCE)

Estimation of the unit-cost equation

    ln(c_u / w_l) = alpha * ln(C_k / w_l) + gamma' X + e,        X = ln knowledge stocks

for the formulations of `estimation_all_bases.ipynb`, with four methods (note "Econometric strategy"), by sector and on pooled groups of sectors. The code follows `estimations_all.py` (Estim_TFP).

**Methods**

| Method | What it does |
| --- | --- |
| Country FE | country fixed effects (country x sector in the pooled estimations) + a dummy for 2009 |
| Country FE + year FE | the same with year fixed effects (the 2009 dummy is then absorbed) |
| CCEP | CCE pooled (Pesaran 2006): every unit is purged of a constant and of the cross-section averages (by sector and year) of y and of every regressor; the residuals of all the units are pooled and one coefficient is estimated |
| CCEMG | CCE mean group: one regression per unit with the same averages; mean of the coefficients of the units (5 % trimmed on each side), standard error = standard deviation / square root of the number of units |

Standard errors of Country FE, Country FE + year FE and CCEP: clustered by country.

**Formulations** (the stocks are in logs): (1) total; (2) national / foreign; (3) intra / inter-sectoral; (4) four channels; (5) absorptive capacity (total stock, own R&D intensity R/Y in % and their product, with the main terms).

**Samples**: `by_sector` = each sector separately (units = countries); pooled groups (units = country x sector, the coefficient of the relative price of capital specific to each sector): manufacturing, transport, ICT, all sectors, market sectors (the lists are in the options cell). The aggregates "C" and "H" are left out of the groups to avoid double counting. `OECD_ICIO` (2016-2022 only) is estimated on the pooled groups with the two fixed-effects methods only.

- Input: `RESULTS_DIR/regression_panels/panel_{BASE}_F.csv`, produced by `panel_setup` (several bases can be estimated in one run: option `RUNS`).
- Output: an Excel file in `RESULTS_DIR/estimation_results/` (suffix `_cce`; the reference tables are not overwritten) and a csv of all the estimates. Cells: coefficient with stars (*** 1 %, ** 5 %, * 10 %). A negative coefficient means that the knowledge stock lowers the unit cost, i.e. raises TFP.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from linearmodels.panel import PanelOLS
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_WANTED, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, CHOC_TAG
warnings.filterwarnings("ignore")
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")

In [ ]:
# ====== What is estimated ======
RUNS = [(BASE, METRIC_WANTED)]   # (base, metric) estimated, metric = "embod" | "dir" | "embod_investment"; several in one run, for instance
                                 # [("EXIOBASE", "embod"), ("EXIOBASE", "dir"), ("GTAP", "embod"), ("FIGARO", "embod")] (their panels are in RESULTS_DIR/regression_panels/)
ONLY_POOLED = False              # True: only the pooled groups of sectors (much faster, no sector-by-sector estimation)
YEAR_FE_BY_SECTOR = False        # Country FE + year FE in the pooled groups: year effects common to all the sectors of the group (False, as in estimations_all.py)
                                 # or specific to each sector (True)
NO_CCE_BASES = ["OECD_ICIO"]     # bases estimated on the pooled groups with the two fixed-effects methods only (series too short for the CCE)
OUT_TAG = "_cce"                 # suffix of the exported file, so that the reference tables are not overwritten

# ====== Samples ======
MANUF = ["C10_C12", "C13_C15", "C16_C18", "C19", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
TRANSPORT = ["H49", "H50", "H51", "H52"]
ICT = ["H53+J", "I"]            # as requested (NB: in NACE Rev.2, I is accommodation and food services)
OTHER = ["A", "B", "D", "E", "F", "G", "K", "L", "M+N", "O+U", "P", "Q", "R+S+T"]
ALL = MANUF + TRANSPORT + ICT + OTHER      # every elementary sector; the aggregates C and H are left out to avoid double counting
MARKET = [s for s in ALL if s not in ("L", "O+U", "P", "Q")]   # without real estate (imputed rents) and the mostly non-market sectors (output measured by inputs)
GROUPS = {"manufacturing": MANUF, "transport": TRANSPORT, "ICT": ICT, "all sectors": ALL, "market sectors": MARKET}

# ====== Formulations (as in estimation_all_bases.ipynb) ======
BLOCKS = {"intrasectoral_national_know": "rd_same_sector_same_country",
          "intrasectoral_foreign_know": "rd_same_sector_diff_country",
          "intersectoral_national_know": "rd_diff_sector_same_country",
          "intersectoral_foreign_know": "rd_diff_sector_diff_country"}
FORMULATIONS = {"(1) total": ["total_know"],
                "(2) national/foreign": ["national_know", "foreign_know"],
                "(3) intra/inter-sectoral": ["intrasectoral_know", "intersectoral_know"],
                "(4) four channels": list(BLOCKS),
                "(5) absorptive capacity": ["total_know", "own_rd_intensity", "total_know_x_intensity"]}
METHODS = ["Country FE", "Country FE + year FE", "CCEP", "CCEMG"]
MIN_OBS_UNIT = 5          # units with fewer observations are dropped
MIN_T_CCE = 12            # CCE: minimum observations per unit


def panel_file(base):
    """Final full-sample panel of a base, produced by panel_setup."""
    return RESULTS_DIR + f"regression_panels/panel_{base}_F.csv"

In [ ]:
def stars(p):
    return "" if not np.isfinite(p) else "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""


def prepare(base, metric):
    """Panel of a base with y = ln(c_u) - ln(w_l), k = ln(C_k) - ln(w_l) and the knowledge stocks of the formulations (logs, positive values only)."""
    d = pd.read_csv(panel_file(base))
    cands = [c for c in d.columns if c.startswith(f"rd_same_sector_same_country_rd_{metric}_")]
    if metric == "embod":
        cands = [c for c in cands if "investment" not in c]
    suffix = cands[0][len("rd_same_sector_same_country_"):]
    with np.errstate(divide="ignore", invalid="ignore"):
        lg = lambda c: np.log(d[c].where(d[c] > 0))
        d["y"] = np.log(d["c_u"]) - np.log(d["w_l"])
        d["k"] = np.log(d["C_k"]) - np.log(d["w_l"])
        d["total_know"] = lg(f"rd_{metric}")
        d["national_know"], d["foreign_know"] = lg(f"rd_{metric}_D"), lg(f"rd_{metric}_F")
        d["intrasectoral_know"], d["intersectoral_know"] = lg(f"rd_{metric}_I"), lg(f"rd_{metric}_X")
        for name, pre in BLOCKS.items():
            d[name] = lg(f"{pre}_{suffix}")
        d["own_rd_intensity"] = d["int_propre"]
        d["total_know_x_intensity"] = d["own_rd_intensity"] * d["total_know"]
    return d.replace([np.inf, -np.inf], np.nan)


def sample(d, sectors, xs):
    """Complete cases of the sectors, one unit = country x sector, units with at least MIN_OBS_UNIT years."""
    g = d[d["Sector"].isin(sectors)].dropna(subset=["y", "k", "2009_crisis"] + xs).copy()
    g["unit"] = g["Region"] + "|" + g["Sector"]
    g = g[g.groupby("unit")["annee"].transform("size") >= MIN_OBS_UNIT]
    return g


def fe(g, xs, year_fe):
    """Fixed effects of the units (+ year effects if year_fe, else the 2009 dummy), standard errors clustered by country.
    Several sectors: the coefficient of k is specific to each sector (alpha_s)."""
    gi = g.set_index(["unit", "annee"]).sort_index()
    X = gi[xs].copy()
    secs = gi["Sector"].unique()
    if len(secs) > 1:
        for s in secs:
            X[f"k_{s}"] = np.where(gi["Sector"] == s, gi["k"], 0.0)
    else:
        X["k"] = gi["k"]
    if not year_fe:
        X["crisis"] = gi["2009_crisis"]
    X["const"] = 1.0
    X = X.loc[:, X.nunique() > 1].assign(const=1.0)
    by_sector = year_fe and YEAR_FE_BY_SECTOR and len(secs) > 1
    other = pd.DataFrame({"sector_year": pd.Categorical(gi["Sector"] + "_" + gi.index.get_level_values(1).astype(str))}, index=gi.index) if by_sector else None
    res = PanelOLS(gi["y"], X, entity_effects=True, time_effects=year_fe and not by_sector, other_effects=other, drop_absorbed=True, check_rank=False) \
        .fit(cov_type="clustered", clusters=pd.DataFrame({"c": pd.Categorical(gi["Region"]).codes}, index=gi.index))
    return {x: (res.params.get(x, np.nan), res.std_errors.get(x, np.nan), res.pvalues.get(x, np.nan)) for x in xs}, \
        int(res.nobs), gi.index.get_level_values(0).nunique()


def cluster_ols(y, X, groups):
    """OLS with standard errors clustered by `groups` (small-sample factors G / (G - 1) and (n - 1) / (n - k))."""
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    meat = np.zeros((X.shape[1], X.shape[1]))
    for gr in np.unique(groups):
        m = groups == gr
        sg = X[m].T @ e[m]
        meat += np.outer(sg, sg)
    G, (n, k) = len(np.unique(groups)), X.shape
    V = G / (G - 1) * (n - 1) / max(n - k, 1) * XtX_inv @ meat @ XtX_inv
    return b, np.sqrt(np.maximum(np.diag(V), 0)), G


def cce(g, xs):
    """CCEP and CCEMG. The common factors are proxied by the cross-section averages (by sector and year) of y, k and of every stock.
    CCEP: every unit is purged of a constant and of these averages, then the residuals are pooled. CCEMG: one regression per unit, 5 % trimmed mean.
    Returns None if there are too few countries or units."""
    vars_ = ["y", "k"] + xs
    g = g[g.groupby("unit")["annee"].transform("size") >= MIN_T_CCE].copy()
    if g["Region"].nunique() < 5 or g["unit"].nunique() < 5:
        return None
    avg = g.groupby(["Sector", "annee"])[vars_].transform("mean")
    bars = []
    for v in vars_:
        g[f"bar_{v}"] = avg[v]
        bars.append(f"bar_{v}")
    parts, mg = [], []
    for u, gu in g.groupby("unit"):
        H = np.column_stack([np.ones(len(gu))] + [gu[b].to_numpy() for b in bars])
        P = np.linalg.pinv(H)
        R = {v: gu[v].to_numpy() - H @ (P @ gu[v].to_numpy()) for v in vars_}
        parts.append(pd.DataFrame({**R, "Sector": gu["Sector"].to_numpy(), "Region": gu["Region"].to_numpy()}))
        X = np.column_stack([np.ones(len(gu)), gu["k"].to_numpy()] + [gu[x].to_numpy() for x in xs] +
                            [gu[b].to_numpy() for b in bars])
        if len(gu) >= X.shape[1] + 5:
            bu = np.linalg.lstsq(X, gu["y"].to_numpy(), rcond=None)[0]
            mg.append(bu[2:2 + len(xs)])
    R = pd.concat(parts, ignore_index=True)
    secs = sorted(R["Sector"].unique())
    kcols = [np.where(R["Sector"] == s, R["k"], 0.0) for s in secs] if len(secs) > 1 else [R["k"].to_numpy()]
    X = np.column_stack([R[x].to_numpy() for x in xs] + kcols)
    b, se, G = cluster_ols(R["y"].to_numpy(), X, R["Region"].to_numpy())
    out = {}
    mg = np.array(mg) if mg else np.empty((0, len(xs)))
    for j, x in enumerate(xs):
        p = 2 * (1 - stats.t.cdf(abs(b[j] / se[j]), G - 1)) if se[j] > 0 else np.nan
        res = {"CCEP": (b[j], se[j], p)}
        c = mg[:, j][np.isfinite(mg[:, j])] if len(mg) else np.array([])
        if len(c) >= 5:
            lo, hi = np.quantile(c, [0.05, 0.95])
            ct = c[(c >= lo) & (c <= hi)]
            s = ct.std(ddof=1) / np.sqrt(len(ct))
            res["CCEMG"] = (ct.mean(), s, 2 * (1 - stats.norm.cdf(abs(ct.mean() / s))))
        else:
            res["CCEMG"] = (np.nan, np.nan, np.nan)
        out[x] = res
    return out, len(g), g["unit"].nunique(), len(mg)

In [ ]:
rows, failures = [], []


def run_sample(label, base, metric, d, sectors, sector_name, with_cce=True):
    """Estimates every formulation on one sample (one sector or one group of sectors) and appends the estimates to `rows`."""
    for form, xs in FORMULATIONS.items():
        g = sample(d, sectors, xs)
        if g["Region"].nunique() < 3 or len(g) < 30:
            continue
        common = dict(sample=label, sector=sector_name, base=base, metric=metric, formulation=form)
        for meth, yfe in [("Country FE", False), ("Country FE + year FE", True)]:
            try:
                r, n, u = fe(g, xs, yfe)
            except Exception as e:
                failures.append((base, metric, sector_name, form, meth, repr(e)))
                continue
            for x, (b, s, p) in r.items():
                rows.append({**common, "variable": x, "method": meth, "coef": b, "se": s, "pvalue": p, "nobs": n, "units": u})
        if with_cce:
            try:
                out = cce(g, xs)
            except Exception as e:
                failures.append((base, metric, sector_name, form, "CCE", repr(e)))
                out = None
            if out is not None:
                r, n, u, n_mg = out
                for x, dct in r.items():
                    for meth in ("CCEP", "CCEMG"):
                        b, s, p = dct[meth]
                        rows.append({**common, "variable": x, "method": meth, "coef": b, "se": s, "pvalue": p,
                                     "nobs": n, "units": u if meth == "CCEP" else n_mg})


for base, metric in RUNS:
    if not os.path.exists(panel_file(base)):
        print(f"[{base}] panel not found: {panel_file(base)}: skipped")
        continue
    d = prepare(base, metric)
    no_cce = base in NO_CCE_BASES
    if not no_cce and not ONLY_POOLED:
        for sector in sorted(d["Sector"].dropna().unique()):
            run_sample("by_sector", base, metric, d, [sector], sector)
    for gname, secs in GROUPS.items():
        run_sample("pooled", base, metric, d, secs, gname, with_cce=not no_cce)
    print("done", base, metric, flush=True)

res = pd.DataFrame(rows)
if res.empty:
    raise RuntimeError("no estimate: check RUNS and the panels in RESULTS_DIR/regression_panels/")
res["cell"] = [("" if not np.isfinite(b) else f"{b:.3f}{stars(p)}") for b, p in zip(res["coef"], res["pvalue"])]
print(f"{len(res)} estimates" + (f", {len(failures)} estimations failed (see `failures`)" if failures else ""))

In [ ]:
form_order = list(FORMULATIONS)
var_order = list(dict.fromkeys(v for xs in FORMULATIONS.values() for v in xs))


def wide(sub, index):
    """One row per (index), one column per method, cells = "coef stars"; observations and units of the Country FE estimation."""
    t = sub.pivot_table(index=index, columns="method", values="cell", aggfunc="first")
    t = t.reindex(columns=[m for m in METHODS if m in t.columns])
    n = sub[sub.method == "Country FE"].groupby(index)[["nobs", "units"]].first()
    t = t.join(n.rename(columns={"nobs": "obs (FE)", "units": "units (FE)"}))
    t = t.reset_index()
    t["formulation"] = pd.Categorical(t["formulation"], form_order, ordered=True)
    t["variable"] = pd.Categorical(t["variable"], var_order, ordered=True)
    return t.sort_values([c for c in index if c in ("sector",)] + ["formulation", "variable", "base", "metric"])


def readme_rows():
    from datetime import datetime
    return [
        "Estimation of the unit-cost equation  ln(c_u / w_l) = alpha * ln(C_k / w_l) + gamma' X + e,  X = ln knowledge stocks.",
        "Methods: " + "; ".join(METHODS) + ". Country FE: country fixed effects (country x sector in the pooled estimations) + 2009 dummy; "
        "Country FE + year FE: with year fixed effects" + (" specific to each sector" if YEAR_FE_BY_SECTOR else " common to all the sectors of the group (pooled groups)") + "; "
        "CCEP / CCEMG: Common Correlated Effects (Pesaran 2006), pooled / mean of the units, 5 % trimmed.",
        "Standard errors: clustered by country (Country FE, year FE, CCEP); CCEMG: standard deviation of the units / sqrt(number of units).",
        "Formulations: " + "; ".join(f"{f} = {', '.join(xs)}" for f, xs in FORMULATIONS.items()) + ".",
        "Samples: by_sector (units = countries); pooled groups (units = country x sector, alpha specific to each sector): "
        + "; ".join(f"{g} = {len(s)} sectors" for g, s in GROUPS.items()) + ". The aggregates C and H are left out.",
        f"Bases and metrics: {RUNS}; bases without CCE (pooled groups, two fixed-effects methods only): {NO_CCE_BASES}.",
        f"Units with fewer than {MIN_OBS_UNIT} observations are dropped; CCE: at least {MIN_T_CCE} observations per unit, 5 countries and 5 units.",
        "",
        "Cells: coefficient with significance stars (*** 1 %, ** 5 %, * 10 %).",
        "Sign: a NEGATIVE coefficient means that the knowledge stock LOWERS the unit cost, i.e. RAISES TFP.",
        "Sheet 'synthesis': all metrics (rows) for the pooled groups and the methods (columns).",
        "Empty CCE cells: too few countries (< 5) or too short series; CCEMG also needs enough observations per unit for the unit regressions.",
        "Sheet 'raw': coefficient, standard error, p-value, observations and units for every estimate.",
        f"Generated {datetime.now():%Y-%m-%d %H:%M}.",
    ]


def export_cce(res, xlsx_path):
    """Excel file: README, synthesis, sample_sizes, one sheet per pooled group, by_sector, raw; csv of all the estimates next to it."""
    res.to_csv(xlsx_path.replace(".xlsx", ".csv"), index=False)
    pool = res[res["sample"] == "pooled"].copy()
    with pd.ExcelWriter(xlsx_path) as w:
        pd.DataFrame({"README": readme_rows()}).to_excel(w, sheet_name="README", index=False)
        syn = pool.pivot_table(index=["formulation", "variable", "base", "metric"], columns=["sector", "method"], values="cell", aggfunc="first")
        syn = syn.reindex(columns=pd.MultiIndex.from_product([list(GROUPS), METHODS])).reset_index()
        syn["formulation"] = pd.Categorical(syn["formulation"], form_order, ordered=True)
        syn["variable"] = pd.Categorical(syn["variable"], var_order, ordered=True)
        syn = syn.sort_values(["formulation", "variable", "base", "metric"]).set_index(["formulation", "variable", "base", "metric"])
        syn.columns = [f"{g} | {m}" for g, m in syn.columns]
        syn = syn.reset_index()
        syn.to_excel(w, sheet_name="synthesis", index=False)
        obs = pool[pool["method"] == "Country FE"].pivot_table(index=["base", "metric"], columns="sector", values=["nobs", "units"], aggfunc="first")
        obs.to_excel(w, sheet_name="sample_sizes")
        for gname in GROUPS:
            wide(pool[pool["sector"] == gname], ["formulation", "variable", "base", "metric"]).to_excel(w, sheet_name=f"pooled_{gname}", index=False)
        if (res["sample"] == "by_sector").any():
            wide(res[res["sample"] == "by_sector"], ["sector", "formulation", "variable", "base", "metric"]).to_excel(w, sheet_name="by_sector", index=False)
        res.drop(columns="cell").to_excel(w, sheet_name="raw", index=False)
    print(f"written {xlsx_path} ({len(res)} estimates)")
    return syn


if len(RUNS) == 1:
    _base, _metric = RUNS[0]
    out_path = RESULTS_DIR + f"estimation_results/{_base}/tab_{_base}_{_metric}{CHOC_TAG}{OUT_TAG}.xlsx"
else:
    out_path = RESULTS_DIR + f"estimation_results/estimations{CHOC_TAG}{OUT_TAG}.xlsx"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
synthesis = export_cce(res, out_path)
synthesis